# IOAI — 2026 Round2 Forda (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
os.makedirs('data', exist_ok=True)
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-round2-forda'
for f in ['train.csv','test.csv']:
    if not os.path.exists(f'data/{f}'): os.system(f'wget -q {BASE}/{f} -O data/{f}')
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# FordA — 모범답안

원 시계열 500점을 feature 로 직접 쓰되, 비선형 경계를 잡는 **HistGradientBoosting** 으로
macro-F1 ≈ 0.50(선형) → **0.81**. (더 끌어올리려면 1D-CNN·ROCKET 특징·트랜스포머.)


In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score

train = pd.read_csv("data/train.csv"); test = pd.read_csv("data/test.csv")
F = [c for c in test.columns if c.startswith("f")]

clf = HistGradientBoostingClassifier(max_iter=200, random_state=0).fit(train[F], train["label"])
print("train macro-F1:", round(f1_score(train["label"], clf.predict(train[F]), average="macro"), 4))

pred = clf.predict(test[F])
pd.DataFrame({"sample_id": test["sample_id"], "label": pred.astype(int)}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(pred))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)